### CUDA error management

In [31]:
%%writefile cuda_stuff.cuh
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <cuda_runtime.h>

#ifndef cuda_stuff_H
#define cuda_stuff_H

//MACRO TO DEBUG CUDA FUNCTIONS
/** Error checking,
 *  taken from https://stackoverflow.com/questions/14038589/what-is-the-canonical-way-to-check-for-errors-using-the-cuda-runtime-api
 */
#define gpuErrchk(ans) { gpuAssert((ans), __FILE__, __LINE__); }
inline void gpuAssert(cudaError_t code, const char *file, int line, bool abort=true)
{
   if (code != cudaSuccess)
   {
      fprintf(stderr,"GPUassert: %s %s %d\n", cudaGetErrorString(code), file, line);
      if (abort) exit(code);
   }
}

#endif

Overwriting cuda_stuff.cuh


In [32]:
%%writefile saxpy.cu
/*
 * GPU code of SAPXPY
 * Y = a.X + Y
 */

#include <stdlib.h>
#include <stdio.h>
#include <cuda.h>
#include <math.h>

#include "cuda_stuff.cuh"

////////////////////////////////////////////////////////////////
//     Vector initialization
////////////////////////////////////////////////////////////////
void init_tab(float *tab, int len, float val) {
    for (int k = 0; k < len; k++) {
       tab[k] = val;
    }
}

void print_tab(const char *tab_name, float *tab, int len){
   int k;
   printf("\n 10 first elements of %s: \n", tab_name);
   for (k = 0; k < 10; k++) {
      printf("%.2f ", tab[k]);
   }

   printf("\n 10 lasts : \n");
   for (k = len - 10; k < len; k++) {
      printf("%.2f ", tab[k]);
   }
   printf("\n");
}



////////////////////////////////////////////////////////////////
//     SAXPY kernel
////////////////////////////////////////////////////////////////
__global__ void saxpy(float *tabX, float *tabY, int len, float a){
   int idx = blockIdx.x * blockDim.x + threadIdx.x;

   if (idx < len) {
      tabY[idx] = a * tabX[idx] + tabY[idx];
   }
}


////////////////////////////////////////////////////////////////
//     Main program
////////////////////////////////////////////////////////////////
int main(int argc, char** argv) {
    float *tabX_d, *tabX_h;  // device and host memory for X
    float *tabY_d, *tabY_h;  // device and host memory for Y
    int len = 1000;

     /** Initialization of the grid **/
    int threadsPerBlock = 256;
    int blocksPerGrid = (len + threadsPerBlock - 1) / threadsPerBlock;
    dim3 grid(blocksPerGrid);
    dim3 block(threadsPerBlock);

    /** Allocation in host memory **/
    tabX_h = (float *) malloc(sizeof(float) * len);
    init_tab(tabX_h, len , 5.);

    tabY_h = (float *) malloc(sizeof(float) * len);
    init_tab(tabY_h, len, 4.);

    /** Allocation in device memory **/
    cudaMalloc((void**) &tabX_d, sizeof(float) * len);
    cudaMalloc((void**) &tabY_d, sizeof(float) * len);

    /** Pre-print of tabY **/
    printf("Before computation \n");
    print_tab("tabY_h", tabY_h, len);

    /** Transfer of data from host to device **/
    cudaMemcpy(tabY_d, tabY_h, len * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(tabX_d, tabX_h, len * sizeof(float), cudaMemcpyHostToDevice);

    /** Create CUDA events for timing **/
    cudaEvent_t start, stop;
    float milliseconds = 0;
    gpuErrchk(cudaEventCreate(&start));
    gpuErrchk(cudaEventCreate(&stop));

    /** SaxPY kernel launching **/
    cudaEventRecord(start);
    saxpy <<< grid, block >>> ( tabX_d, tabY_d, len, 2.0 );
    gpuErrchk(cudaGetLastError());
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("\n[Kernel execution time]: %f ms\n", milliseconds);

    gpuErrchk(cudaPeekAtLastError());
    gpuErrchk(cudaDeviceSynchronize());

    /** Transfer of the result from device to host **/
    gpuErrchk(cudaMemcpy(tabY_h, tabY_d, len * sizeof(float), cudaMemcpyDeviceToHost));

    /** Affichage du resultat **/
    printf("\nAfter computation\n");
    print_tab("tabY_h", tabY_h, len);

    /** Memory free **/
    cudaFree(tabX_d);
    cudaFree(tabY_d);
    free(tabX_h);
    free(tabY_h);

    return EXIT_SUCCESS;
}

Overwriting saxpy.cu


In [33]:
! nvcc -arch sm_75 saxpy.cu -o saxpy

In [34]:
! ./saxpy

Before computation 

 10 first elements of tabY_h: 
4.00 4.00 4.00 4.00 4.00 4.00 4.00 4.00 4.00 4.00 
 10 lasts : 
4.00 4.00 4.00 4.00 4.00 4.00 4.00 4.00 4.00 4.00 

[Kernel execution time]: 0.136832 ms

After computation

 10 first elements of tabY_h: 
14.00 14.00 14.00 14.00 14.00 14.00 14.00 14.00 14.00 14.00 
 10 lasts : 
14.00 14.00 14.00 14.00 14.00 14.00 14.00 14.00 14.00 14.00 
